# Local Eval Harness v4 — Chat Template Fix

**Hypothesis:** v3's 0/10 was caused by feeding raw prompts to an instruction-tuned model. With proper `apply_chat_template`, the model should:
- No longer hallucinate the 'Please reason step by step...' prefix
- No longer hit 16-token early-stop EOS
- Produce reasoning inside `<think>...</think>` and final answer after

**Format (from Nemotron 3 chat_template.jinja):**
```
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
<think>
{reasoning}
</think>
{final answer}<|im_end|>
```

**Extraction:** content between `</think>` and `<|im_end|>` → strip → that's the answer.
**Stopping:** stop on `<|im_end|>` (token id 11).
**HOLDOUT_N=3** for first run — verify the pipeline before scaling. If timing is reasonable, v5 ramps to N=10-20.

In [ ]:
# Cell 1: Config + setup (same as v3)
import os, random, shutil, site, stat, sys, time

import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
HOLDOUT_N = 3                    # tiny — verify pipeline works before scaling
MAX_NEW_TOKENS = 1024

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
step(f"Config: HOLDOUT_N={HOLDOUT_N}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}")

# sitedir discovery + ptxas-blackwell exec workaround
for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break
step("Setup done")

In [ ]:
# Cell 2: Load model + tokenizer + LoRA + verify chat template
import kagglehub
import mamba_ssm  # noqa: F401
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
step("Tokenizer loaded")

# Verify chat template is present and special tokens look right
assert tokenizer.chat_template is not None, "chat_template missing — model may not be the expected variant"
step(f"Chat template present ({len(tokenizer.chat_template)} chars)")

# Find <|im_end|> token id (we'll use as EOS for chat completion)
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
THINK_OPEN_ID = tokenizer.convert_tokens_to_ids("<think>")
THINK_CLOSE_ID = tokenizer.convert_tokens_to_ids("</think>")
step(f"Special tokens: <|im_end|>={IM_END_ID}, <think>={THINK_OPEN_ID}, </think>={THINK_CLOSE_ID}")

# Sanity-print what apply_chat_template produces for a sample prompt
sample_messages = [{"role": "user", "content": "What is 2+2? Answer with just the number."}]
sample_text = tokenizer.apply_chat_template(
    sample_messages, tokenize=False, add_generation_prompt=True
)
step("Sample formatted prompt:")
print(repr(sample_text))

# Now load model
t = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
model.eval()
step(f"Model loaded in {time.perf_counter() - t:.0f}s")

lora_cfg = LoraConfig(
    r=32, lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
step("LoRA wrapped (untrained)")

In [ ]:
# Cell 3: Extraction helper — parse </think>{answer}<|im_end|>
import re

THINK_CLOSE_RE = re.compile(r"</think>")
IM_END_RE = re.compile(r"<\|im_end\|>")

def extract_chat_answer(completion_text: str) -> dict:
    """
    Returns {
      'reasoning': content between <think> (implicit start) and </think>,
      'answer': content after </think>, before <|im_end|> (or end of string),
      'has_think_close': whether </think> was produced,
    }
    """
    has_close = bool(THINK_CLOSE_RE.search(completion_text))
    if has_close:
        parts = THINK_CLOSE_RE.split(completion_text, maxsplit=1)
        reasoning = parts[0].strip()
        post = parts[1]
        # Trim at <|im_end|> if present
        end_match = IM_END_RE.search(post)
        if end_match:
            answer = post[: end_match.start()].strip()
        else:
            answer = post.strip()
    else:
        reasoning = completion_text.strip()
        answer = ""
    return {"reasoning": reasoning, "answer": answer, "has_think_close": has_close}

# Self-test
t = extract_chat_answer("some reasoning</think>42<|im_end|>")
assert t["answer"] == "42" and t["has_think_close"], t
t = extract_chat_answer("unfinished reasoning")
assert t["answer"] == "" and not t["has_think_close"], t
step("Extraction helper ready")

In [ ]:
# Cell 4: Held-out slice (last HOLDOUT_N by id sort, same as v3 for comparability)
import pandas as pd

TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV).sort_values("id").reset_index(drop=True)
holdout = df.tail(HOLDOUT_N).reset_index(drop=True)
step(f"Held-out: {holdout.shape} (last {HOLDOUT_N}; first id={holdout.iloc[0]['id']}, last id={holdout.iloc[-1]['id']})")

In [ ]:
# Cell 5: Inference loop — apply_chat_template + stop on <|im_end|>
results = []
t_loop = time.perf_counter()

for i, row in holdout.iterrows():
    pid = row["id"]
    prompt = row["prompt"]
    gold = str(row["answer"])

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True,
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    t_p = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            # Stop on <|im_end|> OR </s> (model's EOS variants)
            eos_token_id=[IM_END_ID, tokenizer.eos_token_id],
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - t_p
    n_new = out.shape[1] - prompt_len
    completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=False)  # KEEP special tokens for extraction
    parsed = extract_chat_answer(completion)
    correct = parsed["answer"].strip() == gold.strip()

    results.append({
        "id": pid,
        "prompt_len_tokens": int(prompt_len),
        "completion_tokens": int(n_new),
        "completion": completion,
        "reasoning": parsed["reasoning"],
        "answer": parsed["answer"],
        "has_think_close": parsed["has_think_close"],
        "gold": gold,
        "correct": correct,
        "seconds": elapsed,
    })
    print(
        f"  [{i+1}/{HOLDOUT_N}] id={pid} | {n_new:4d} tok in {elapsed:5.1f}s | "
        f"close={parsed['has_think_close']} | answer={parsed['answer']!r:25s} | gold={gold!r:15s} | "
        f"{'✓' if correct else '✗'}",
        flush=True,
    )

step(f"Inference loop done in {time.perf_counter() - t_loop:.0f}s")

In [ ]:
# Cell 6: Summary + save
import json

n = len(results)
n_correct = sum(r["correct"] for r in results)
n_think_closed = sum(1 for r in results if r["has_think_close"])
avg_tokens = sum(r["completion_tokens"] for r in results) / max(n, 1)
avg_secs = sum(r["seconds"] for r in results) / max(n, 1)

summary = {
    "holdout_n": n,
    "version": "v4-chat-template",
    "max_new_tokens": MAX_NEW_TOKENS,
    "adapter": "untrained (= base model)",
    "accuracy": n_correct / max(n, 1),
    "correct": n_correct,
    "think_closed_rate": n_think_closed / max(n, 1),
    "avg_completion_tokens": avg_tokens,
    "avg_seconds_per_problem": avg_secs,
    "total_inference_seconds": sum(r["seconds"] for r in results),
}

out_df = pd.DataFrame(results)
out_df.to_csv("/kaggle/working/eval_results.csv", index=False)
with open("/kaggle/working/eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n=== Summary ===")
print(json.dumps(summary, indent=2))

In [ ]:
# Cell 7: Print full output of each problem so we can SEE what changed vs v3
for i, r in enumerate(results, 1):
    print(f"=== Problem {i} (id={r['id']}, gold={r['gold']!r}) ===")
    print(f"answer extracted: {r['answer']!r}")
    print(f"think_close: {r['has_think_close']}, completion_tokens: {r['completion_tokens']}")
    print("--- COMPLETION (first 800 chars) ---")
    print(r["completion"][:800])
    if len(r["completion"]) > 800:
        print(f"... [+{len(r['completion']) - 800} more chars]")
    print()
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")